# Eksperimen 11 — 5-Fold Cross Validation pada Seluruh Skenario Ablasi

## Deskripsi

Notebook ini digunakan untuk mengevaluasi performa 5 skenario arsitektur model menggunakan **Stratified 5-Fold Cross Validation (K = 5)** sehingga diperoleh estimasi performa model yang lebih robust dan tidak bergantung pada satu pembagian data (*train-validation split*).

---

## Skenario Eksperimen

| No | Skenario | Modul Aktif |
|----|----------|-------------|
| 1 | Baseline 3D-CNN | 3D-CNN murni tanpa mekanisme attention |
| 2 | Ablasi A — Tanpa Prior | Learned Spatial + Temporal (BSP dinonaktifkan) |
| 3 | Ablasi B — Tanpa Learned Spatial | BSP + Temporal (Learned Spatial dinonaktifkan) |
| 4 | Ablasi C — Tanpa Temporal | BSP + Learned Spatial (Temporal dinonaktifkan) |
| 5 | Full AttentiveSkel-3D | BSP + Learned Spatial + Temporal |

---

## Protokol Eksperimen

- Menggunakan **Stratified K-Fold Cross Validation (K = 5)**.
- Model diinisialisasi ulang pada setiap fold sehingga bobot tidak diwariskan antar fold.
- Metrik utama yang dicatat adalah **Best Validation Accuracy**, yaitu akurasi validasi terbaik berdasarkan nilai *validation loss* terendah.
- Optimizer menggunakan **Adam** (`learning rate = 1e-3`, `weight_decay = 1e-4`).
- Learning rate disesuaikan menggunakan scheduler **ReduceLROnPlateau**.
- Distribusi kelas dijaga tetap seimbang pada setiap fold menggunakan **StratifiedKFold**.

# 1. Setup dan Import Library

Pada tahap ini dilakukan:

- Menambahkan direktori root proyek ke dalam `PYTHONPATH`.
- Mengimpor seluruh library yang diperlukan.
- Menampilkan informasi lingkungan (*environment*) yang digunakan selama eksperimen.

In [18]:
import sys
from pathlib import Path

# =============================================================================
# Menambahkan root project ke PYTHONPATH
# =============================================================================

ROOT_DIR = Path("..").resolve()

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# =============================================================================
# Import Library
# =============================================================================

import torch
import pandas as pd

from src.models.kfold_training import (
    run_kfold_experiment,
    SCENARIOS,
)

# =============================================================================
# Informasi Environment
# =============================================================================

print(f"Root Proyek     : {ROOT_DIR}")
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

print(f"\nJumlah skenario yang akan diuji: {len(SCENARIOS)}")

for i, scenario in enumerate(SCENARIOS, start=1):
    print(f"{i}. {scenario}")

Root Proyek     : D:\Data-Aji\KULIAH\Tugas-Akhir\AttentiveSkel3D-WeightTraining-PoC
PyTorch Version : 2.5.1
CUDA Available  : True
GPU             : NVIDIA GeForce RTX 3060 Ti

Jumlah skenario yang akan diuji: 5
1. Baseline 3D-CNN
2. Ablasi A — Tanpa Prior
3. Ablasi B — Tanpa Learned Spatial
4. Ablasi C — Tanpa Temporal
5. Full AttentiveSkel-3D


# 2. Konfigurasi Eksperimen

Parameter berikut digunakan selama proses pelatihan dan evaluasi model.

Apabila diperlukan, nilai parameter dapat diubah sebelum eksperimen dijalankan.

In [19]:
# =============================================================================
# Konfigurasi Eksperimen
# =============================================================================

MANIFEST_PATH = ROOT_DIR / "data" / "processed" / "dataset_manifest.csv"

CONFIG = dict(
    dataset_manifest_path=MANIFEST_PATH,
    num_folds=5,
    epochs=100,               # Naikkan menjadi 100 epoch untuk eksperimen final
    batch_size=16,
    device="cuda",           # Otomatis menggunakan CPU apabila CUDA tidak tersedia
    lr=1e-3,
    num_workers=0,           # Direkomendasikan 0 pada Windows
    random_state=42,
    verbose=True,
)

# =============================================================================
# Menampilkan Konfigurasi
# =============================================================================

print("Konfigurasi Eksperimen\n")

for key, value in CONFIG.items():
    print(f"{key:<28}: {value}")

print(f"\nDataset Manifest tersedia : {MANIFEST_PATH.exists()}")
print(f"Lokasi Manifest           : {MANIFEST_PATH}")

Konfigurasi Eksperimen

dataset_manifest_path       : D:\Data-Aji\KULIAH\Tugas-Akhir\AttentiveSkel3D-WeightTraining-PoC\data\processed\dataset_manifest.csv
num_folds                   : 5
epochs                      : 100
batch_size                  : 16
device                      : cuda
lr                          : 0.001
num_workers                 : 0
random_state                : 42
verbose                     : True

Dataset Manifest tersedia : True
Lokasi Manifest           : D:\Data-Aji\KULIAH\Tugas-Akhir\AttentiveSkel3D-WeightTraining-PoC\data\processed\dataset_manifest.csv


# 3. Pelaksanaan 5-Fold Cross Validation

Pada tahap ini dilakukan proses pelatihan dan evaluasi model menggunakan **Stratified 5-Fold Cross Validation** pada seluruh skenario arsitektur yang telah didefinisikan sebelumnya.

Setiap fold akan:

- Menginisialisasi model dari awal (*training from scratch*).
- Melatih model menggunakan data latih (*training set*).
- Mengevaluasi model menggunakan data validasi (*validation set*).
- Menyimpan nilai **Best Validation Accuracy** berdasarkan *validation loss* terbaik.

> **Estimasi Waktu**
>
> - CPU : ± 20–40 menit
> - GPU : ± 2–5 menit per skenario
>
> Total 5 skenario × 5 fold × N epoch. Progres dicetak setiap 10 epoch.
>
> Lama proses bergantung pada spesifikasi perangkat, jumlah epoch, dan ukuran dataset.


In [20]:
# =============================================================================
# Menjalankan Eksperimen 5-Fold Cross Validation
# =============================================================================

kfold_df = run_kfold_experiment(**CONFIG)

  5-Fold Cross Validation — AttentiveSkel-3D
  Manifest  : D:\Data-Aji\KULIAH\Tugas-Akhir\AttentiveSkel3D-WeightTraining-PoC\data\processed\dataset_manifest.csv
  Device    : cuda  (diminta: cuda)
  Folds     : 5
  Epochs    : 100
  Batch size: 16
  LR (Adam) : 0.001

  Total sampel dataset : 487
  Distribusi label     : kelas 0=303, kelas 1=184

──────────────────────────────────────────────────────────────
  Skenario : Baseline 3D-CNN
──────────────────────────────────────────────────────────────


c:\Users\Administrator\anaconda3\envs\attentiveskel\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


    Fold 1/5 | Epoch   1/100 | TrainLoss: 0.4797  TrainAcc: 0.799 | ValLoss: 0.4594  ValAcc: 0.827  [Best ValAcc: 0.827]
    Fold 1/5 | Epoch  10/100 | TrainLoss: 0.1766  TrainAcc: 0.946 | ValLoss: 0.3682  ValAcc: 0.878  [Best ValAcc: 0.908]
    Fold 1/5 | Epoch  20/100 | TrainLoss: 0.1587  TrainAcc: 0.946 | ValLoss: 0.3173  ValAcc: 0.888  [Best ValAcc: 0.918]
    Fold 1/5 | Epoch  30/100 | TrainLoss: 0.0723  TrainAcc: 0.972 | ValLoss: 0.2979  ValAcc: 0.918  [Best ValAcc: 0.908]
    Fold 1/5 | Epoch  40/100 | TrainLoss: 0.0405  TrainAcc: 0.985 | ValLoss: 0.4086  ValAcc: 0.939  [Best ValAcc: 0.908]
    Fold 1/5 | Epoch  50/100 | TrainLoss: 0.0366  TrainAcc: 0.985 | ValLoss: 0.3976  ValAcc: 0.939  [Best ValAcc: 0.908]
    Fold 1/5 | Epoch  60/100 | TrainLoss: 0.0222  TrainAcc: 0.992 | ValLoss: 0.4386  ValAcc: 0.929  [Best ValAcc: 0.908]
    Fold 1/5 | Epoch  70/100 | TrainLoss: 0.0193  TrainAcc: 0.992 | ValLoss: 0.5163  ValAcc: 0.929  [Best ValAcc: 0.908]
    Fold 1/5 | Epoch  80/100 | T

# 4. Hasil 5-Fold Cross Validation

Bagian ini menyajikan hasil evaluasi setiap skenario model pada lima fold.

Nilai yang ditampilkan merupakan **Best Validation Accuracy** dari masing-masing fold, kemudian dihitung nilai:

- Mean Accuracy
- Standard Deviation

Agar lebih mudah dianalisis, hasil ditampilkan dalam bentuk **heatmap** sehingga model dengan performa terbaik dapat langsung diidentifikasi.

In [21]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# =============================================================================
# Menentukan Kolom Fold
# =============================================================================

fold_cols = [col for col in kfold_df.columns if col.startswith("Fold")]

# =============================================================================
# Konversi Nilai ke Persentase
# =============================================================================

display_df = kfold_df.copy()

for col in fold_cols + ["Mean Accuracy", "Std Deviation"]:
    display_df[col] = (display_df[col] * 100).round(2)

# =============================================================================
# Mengubah Nama Kolom
# =============================================================================

display_df = display_df.rename(
    columns={
        "Mean Accuracy": "Mean Acc (%)",
        "Std Deviation": "Std Dev (%)",
    }
)

fold_display_cols = fold_cols + ["Mean Acc (%)"]

# =============================================================================
# Styling DataFrame
# =============================================================================

styled = (
    display_df.style
    .background_gradient(
        cmap="Greens",
        subset=fold_display_cols,
        vmin=50,
        vmax=100,
    )
    .background_gradient(
        cmap="Reds_r",
        subset=["Std Dev (%)"],
        vmin=0,
        vmax=10,
    )
    .format(
        "{:.2f}%",
        subset=fold_display_cols + ["Std Dev (%)"],
    )
    .set_caption(
        "Hasil 5-Fold Cross Validation — Best Validation Accuracy (%)"
    )
    .set_table_styles(
        [
            {
                "selector": "caption",
                "props": [
                    ("font-size", "14px"),
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                    ("padding", "8px 0"),
                ],
            },
            {
                "selector": "th",
                "props": [
                    ("font-size", "12px"),
                    ("text-align", "center"),
                    ("padding", "8px 12px"),
                    ("background-color", "#1a2b3c"),
                    ("color", "#c0d4ec"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("font-size", "12px"),
                    ("text-align", "center"),
                    ("padding", "7px 12px"),
                ],
            },
            {
                "selector": "td:first-child",
                "props": [
                    ("text-align", "left"),
                    ("font-weight", "600"),
                    ("padding-left", "14px"),
                    ("white-space", "nowrap"),
                ],
            },
        ]
    )
    .highlight_max(
        subset=["Mean Acc (%)"],
        color="#00c853",
        axis=0,
    )
    .highlight_min(
        subset=["Mean Acc (%)"],
        color="#ff5252",
        axis=0,
    )
)

display(styled)

,Skenario,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean Acc (%),Std Dev (%)
0,Baseline 3D-CNN,90.82%,92.86%,98.97%,90.72%,98.97%,94.47%,4.20%
1,Ablasi A — Tanpa Prior,93.88%,90.82%,97.94%,93.81%,100.00%,95.29%,3.65%
2,Ablasi B — Tanpa Learned Spatial,91.84%,94.90%,94.85%,93.81%,100.00%,95.08%,3.02%
3,Ablasi C — Tanpa Temporal,93.88%,92.86%,97.94%,87.63%,98.97%,94.25%,4.52%
4,Full AttentiveSkel-3D,92.86%,91.84%,98.97%,92.78%,100.00%,95.29%,3.87%


## Interpretasi Hasil

Tabel di atas menunjukkan performa masing-masing skenario model pada lima fold pengujian.

Beberapa aspek yang dapat diamati meliputi:

- Konsistensi performa model pada setiap fold.
- Nilai **Mean Accuracy** sebagai indikator performa rata-rata.
- Nilai **Standard Deviation** sebagai indikator stabilitas model.
- Perbandingan awal antara model baseline dan model yang menggunakan mekanisme attention.

Analisis yang lebih rinci akan disajikan pada bagian berikutnya melalui proses pemeringkatan (*ranking*) seluruh skenario model.

# 5. Ringkasan Perbandingan Model

Setelah seluruh skenario dievaluasi menggunakan **5-Fold Cross Validation**, langkah berikutnya adalah melakukan peringkat (*ranking*) berdasarkan nilai **Mean Validation Accuracy**.

Selain nilai rata-rata akurasi, juga ditampilkan **Standard Deviation** untuk melihat tingkat konsistensi performa model pada setiap fold.

Bagian ini bertujuan untuk:

- Mengidentifikasi model dengan performa terbaik.
- Mengidentifikasi model dengan performa terendah.
- Membandingkan peningkatan performa model **Full AttentiveSkel-3D** terhadap **Baseline 3D-CNN**.

In [22]:
# =============================================================================
# Menyusun Ranking Model Berdasarkan Mean Accuracy
# =============================================================================

ranked_df = (
    kfold_df[
        ["Skenario", "Mean Accuracy", "Std Deviation"]
    ]
    .copy()
    .sort_values(
        by="Mean Accuracy",
        ascending=False,
    )
    .reset_index(drop=True)
)

# Mulai ranking dari angka 1
ranked_df.index += 1
ranked_df.index.name = "Rank"

# =============================================================================
# Konversi Nilai ke Persentase
# =============================================================================

ranked_df["Mean Accuracy"] = (
    ranked_df["Mean Accuracy"] * 100
).round(2)

ranked_df["Std Deviation"] = (
    ranked_df["Std Deviation"] * 100
).round(2)

# =============================================================================
# Mengubah Nama Kolom
# =============================================================================

ranked_df = ranked_df.rename(
    columns={
        "Skenario": "Skenario (Ranking)",
        "Mean Accuracy": "Mean Acc (%)",
        "Std Deviation": "Std Dev (%)",
    }
)

In [23]:
# =============================================================================
# Styling Ranking Model
# =============================================================================

ranked_styled = (
    ranked_df.style
    .background_gradient(
        cmap="RdYlGn",
        subset=["Mean Acc (%)"],
        vmin=50,
        vmax=100,
    )
    .format(
        "{:.2f}%",
        subset=["Mean Acc (%)", "Std Dev (%)"],
    )
    .set_caption(
        "Ranking Model Berdasarkan Mean Validation Accuracy (5-Fold Cross Validation)"
    )
    .set_table_styles(
        [
            {
                "selector": "caption",
                "props": [
                    ("font-size", "14px"),
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                    ("padding", "8px 0"),
                ],
            },
            {
                "selector": "th",
                "props": [
                    ("font-size", "12px"),
                    ("text-align", "center"),
                    ("padding", "8px 12px"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("font-size", "12px"),
                    ("text-align", "center"),
                    ("padding", "7px 12px"),
                ],
            },
            {
                "selector": "td:first-child",
                "props": [
                    ("text-align", "left"),
                    ("font-weight", "600"),
                    ("padding-left", "14px"),
                ],
            },
        ]
    )
)

display(ranked_styled)

,Skenario (Ranking),Mean Acc (%),Std Dev (%)
Rank,,,
1,Ablasi A — Tanpa Prior,95.29%,3.65%
2,Full AttentiveSkel-3D,95.29%,3.87%
3,Ablasi B — Tanpa Learned Spatial,95.08%,3.02%
4,Baseline 3D-CNN,94.47%,4.20%
5,Ablasi C — Tanpa Temporal,94.25%,4.52%


In [24]:
# =============================================================================
# Interpretasi Hasil Ranking
# =============================================================================

best_model = ranked_df.iloc[0]
worst_model = ranked_df.iloc[-1]

print("\n📊 Ringkasan Hasil")

print(
    f"✅ Model Terbaik : {best_model['Skenario (Ranking)']}"
)

print(
    f"   Mean Accuracy : {best_model['Mean Acc (%)']:.2f}%"
)

print(
    f"   Std Deviation : {best_model['Std Dev (%)']:.2f}%"
)

print()

print(
    f"❌ Model Terendah : {worst_model['Skenario (Ranking)']}"
)

print(
    f"   Mean Accuracy : {worst_model['Mean Acc (%)']:.2f}%"
)

print(
    f"   Std Deviation : {worst_model['Std Dev (%)']:.2f}%"
)


📊 Ringkasan Hasil
✅ Model Terbaik : Ablasi A — Tanpa Prior
   Mean Accuracy : 95.29%
   Std Deviation : 3.65%

❌ Model Terendah : Ablasi C — Tanpa Temporal
   Mean Accuracy : 94.25%
   Std Deviation : 4.52%


In [25]:
# =============================================================================
# Perbandingan Full Model dengan Baseline
# =============================================================================

try:

    full_acc = kfold_df.loc[
        kfold_df["Skenario"] == "Full AttentiveSkel-3D",
        "Mean Accuracy",
    ].values[0]

    baseline_acc = kfold_df.loc[
        kfold_df["Skenario"] == "Baseline 3D-CNN",
        "Mean Accuracy",
    ].values[0]

    delta = (full_acc - baseline_acc) * 100

    sign = "+" if delta >= 0 else ""

    print(
        f"\n📈 Peningkatan Full Model terhadap Baseline : "
        f"{sign}{delta:.2f} poin akurasi"
    )

except (IndexError, KeyError):

    print(
        "\n⚠️ Data Full Model atau Baseline tidak ditemukan."
    )


📈 Peningkatan Full Model terhadap Baseline : +0.82 poin akurasi


## Interpretasi

Berdasarkan hasil pemeringkatan di atas dapat diamati bahwa:

- Model dengan **Mean Accuracy** tertinggi merupakan kandidat terbaik untuk digunakan sebagai model akhir penelitian.
- Nilai **Standard Deviation** yang kecil menunjukkan performa model lebih stabil pada seluruh fold.
- Selisih akurasi antara **Full AttentiveSkel-3D** dan **Baseline 3D-CNN** menunjukkan kontribusi mekanisme attention yang diusulkan terhadap peningkatan performa klasifikasi.

# 6. Menyimpan Hasil Eksperimen

Sebagai dokumentasi penelitian, hasil evaluasi disimpan dalam format **CSV** sehingga dapat digunakan kembali untuk analisis lanjutan maupun penyusunan BAB 4.

In [26]:
# =============================================================================
# Menyimpan Hasil K-Fold Cross Validation
# =============================================================================

OUTPUT_CSV = (
    ROOT_DIR
    / "data"
    / "processed"
    / "100_kfold_results.csv"
)

OUTPUT_CSV.parent.mkdir(
    parents=True,
    exist_ok=True,
)

kfold_df.to_csv(
    OUTPUT_CSV,
    index=False,
)

print("\n✅ Hasil berhasil disimpan")

print(f"Lokasi File : {OUTPUT_CSV}")
print(f"Ukuran      : {OUTPUT_CSV.stat().st_size / 1024:.2f} KB")
print(f"Dimensi     : {kfold_df.shape[0]} × {kfold_df.shape[1]}")


✅ Hasil berhasil disimpan
Lokasi File : D:\Data-Aji\KULIAH\Tugas-Akhir\AttentiveSkel3D-WeightTraining-PoC\data\processed\100_kfold_results.csv
Ukuran      : 0.43 KB
Dimensi     : 5 × 8


# 8. Ekspor Hasil 100 Epoch (CSV & Single-HTML)

Bagian ini mendokumentasikan snapshot hasil evaluasi **5-Fold Cross Validation dengan 100 epoch** sebelum dianalisis lebih lanjut.

Hasil disimpan secara terpisah dari hasil 50 epoch sehingga perbandingan langsung antara kedua konfigurasi dapat dilakukan di kemudian hari.

In [27]:
# =============================================================================
# Ekspor Hasil 100 Epoch ke CSV
# =============================================================================

CSV_SUMMARY_100 = ROOT_DIR / "data" / "processed" / "kfold_100epochs_summary.csv"
CSV_RANKING_100 = ROOT_DIR / "data" / "processed" / "kfold_100epochs_ranking.csv"

CSV_SUMMARY_100.parent.mkdir(parents=True, exist_ok=True)

# Simpan tabel lengkap K-Fold (raw, nilai 0–1)
kfold_df.to_csv(CSV_SUMMARY_100, index=False)

# Rekonstruksi ranked_df versi raw dari kfold_df saat ini (100 epoch)
ranked_df_raw_100 = (
    kfold_df[["Skenario", "Mean Accuracy", "Std Deviation"]]
    .copy()
    .sort_values(by="Mean Accuracy", ascending=False)
    .reset_index(drop=True)
)
ranked_df_raw_100.index += 1
ranked_df_raw_100.index.name = "Rank"

ranked_df_raw_100.to_csv(CSV_RANKING_100, index=True)

print("✅ Hasil 100 Epoch berhasil disimpan ke CSV\n")
print(f"📄 Summary  : {CSV_SUMMARY_100}")
print(f"📄 Ranking  : {CSV_RANKING_100}")
print(f"\n📊 Dimensi tabel summary : {kfold_df.shape[0]} × {kfold_df.shape[1]}")
print(f"📊 Dimensi tabel ranking : {ranked_df_raw_100.shape[0]} × {ranked_df_raw_100.shape[1]}")

✅ Hasil 100 Epoch berhasil disimpan ke CSV

📄 Summary  : D:\Data-Aji\KULIAH\Tugas-Akhir\AttentiveSkel3D-WeightTraining-PoC\data\processed\kfold_100epochs_summary.csv
📄 Ranking  : D:\Data-Aji\KULIAH\Tugas-Akhir\AttentiveSkel3D-WeightTraining-PoC\data\processed\kfold_100epochs_ranking.csv

📊 Dimensi tabel summary : 5 × 8
📊 Dimensi tabel ranking : 5 × 3


In [28]:
# =============================================================================
# Generate Single-HTML Presentation — Hasil 100 Epoch
# =============================================================================

# Konversi DataFrame ke persentase untuk presentasi
kfold_df_pct_100 = kfold_df.copy()
for col in kfold_df_pct_100.columns:
    if col != "Skenario":
        kfold_df_pct_100[col] = (kfold_df_pct_100[col] * 100).round(2)

ranked_df_pct_100 = ranked_df_raw_100.copy()
ranked_df_pct_100["Mean Accuracy"] = (ranked_df_pct_100["Mean Accuracy"] * 100).round(2)
ranked_df_pct_100["Std Deviation"] = (ranked_df_pct_100["Std Deviation"] * 100).round(2)

# Konversi ke HTML
ranking_html_100  = ranked_df_pct_100.to_html(classes="custom-table", index=True,  border=0)
detail_html_100   = kfold_df_pct_100.to_html( classes="custom-table", index=False, border=0)

html_100 = f"""<!DOCTYPE html>
<html lang="id">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Hasil 5-Fold Cross Validation (100 Epochs) - AttentiveSkel-3D</title>
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}

        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #0f0f23 0%, #1a1a2e 100%);
            color: #e0e0e0;
            padding: 40px 20px;
            line-height: 1.6;
        }}

        .container {{
            max-width: 1200px;
            margin: 0 auto;
            background: rgba(22, 27, 34, 0.95);
            border-radius: 16px;
            padding: 40px;
            box-shadow: 0 8px 32px rgba(0, 255, 135, 0.1);
            border: 1px solid rgba(0, 255, 135, 0.2);
        }}

        h1 {{
            color: #00ff87;
            font-size: 2.4em;
            text-align: center;
            margin-bottom: 8px;
            text-shadow: 0 0 20px rgba(0, 255, 135, 0.5);
            font-weight: 700;
        }}

        .subtitle {{
            text-align: center;
            color: #888;
            margin-bottom: 14px;
            font-size: 1.05em;
        }}

        .warning-box {{
            background: rgba(255, 152, 0, 0.1);
            border: 1px solid rgba(255, 152, 0, 0.5);
            border-left: 5px solid #ff9800;
            border-radius: 8px;
            padding: 18px 22px;
            margin: 20px 0 36px 0;
            font-size: 1.0em;
            line-height: 1.7;
        }}

        .warning-box .warn-title {{
            color: #ff9800;
            font-weight: 700;
            font-size: 1.1em;
            margin-bottom: 6px;
        }}

        .warning-box strong {{ color: #ffcc80; }}

        h2 {{
            color: #00d4ff;
            font-size: 1.7em;
            margin: 36px 0 18px 0;
            padding-bottom: 10px;
            border-bottom: 2px solid rgba(0, 212, 255, 0.3);
            font-weight: 600;
        }}

        .narasi {{
            background: rgba(0, 212, 255, 0.05);
            border-left: 4px solid #00d4ff;
            padding: 18px 22px;
            margin: 18px 0;
            border-radius: 8px;
            font-size: 1.02em;
            line-height: 1.8;
        }}

        .narasi strong {{ color: #00ff87; font-weight: 600; }}

        .custom-table {{
            width: 100%;
            border-collapse: collapse;
            margin: 20px 0;
            font-size: 0.95em;
            background: rgba(30, 30, 46, 0.6);
            border-radius: 12px;
            overflow: hidden;
            box-shadow: 0 4px 16px rgba(0, 0, 0, 0.3);
        }}

        .custom-table thead tr {{
            background: linear-gradient(135deg, #1a2332 0%, #0f1419 100%);
            color: #00d4ff;
            text-align: left;
            font-weight: 700;
            text-transform: uppercase;
            font-size: 0.85em;
            letter-spacing: 0.5px;
        }}

        .custom-table th,
        .custom-table td {{
            padding: 15px 20px;
            border-bottom: 1px solid rgba(255, 255, 255, 0.05);
        }}

        .custom-table tbody tr {{ transition: all 0.3s ease; }}

        .custom-table tbody tr:hover {{
            background: rgba(0, 255, 135, 0.08);
            transform: translateX(4px);
        }}

        .custom-table tbody tr:nth-of-type(even) {{
            background: rgba(255, 255, 255, 0.02);
        }}

        .custom-table tbody tr:first-child {{
            background: linear-gradient(90deg, rgba(0, 255, 135, 0.15) 0%, rgba(0, 212, 255, 0.1) 100%);
            font-weight: 600;
        }}

        .custom-table tbody tr:last-child {{
            background: rgba(255, 82, 82, 0.08);
        }}

        .custom-table td:first-child {{ color: #00ff87; font-weight: 600; }}

        footer {{
            text-align: center;
            margin-top: 50px;
            padding-top: 28px;
            border-top: 1px solid rgba(255, 255, 255, 0.1);
            color: #666;
            font-size: 0.9em;
        }}
    </style>
</head>
<body>
    <div class="container">

        <h1>📊 Hasil Evaluasi 5-Fold Cross Validation</h1>
        <p class="subtitle">AttentiveSkel-3D — Eksperimen 100 Epochs</p>

        <div class="warning-box">
            <p class="warn-title">⚠️ Catatan Eksperimen</p>
            <p>Pelatihan <strong>100 epoch</strong> menunjukkan indikasi <strong>Overfitting</strong> dengan
            <strong>Standard Deviasi yang lebih tinggi</strong> dibandingkan hasil 50 epoch. Hal ini mengindikasikan
            bahwa model mulai menghafal pola pada data latih dan kehilangan kemampuan generalisasi pada data validasi
            yang belum pernah dilihat sebelumnya. Konfigurasi <strong>50 epoch</strong> direkomendasikan sebagai
            titik optimal untuk penelitian ini.</p>
        </div>

        <h2>🏆 Ranking Model Berdasarkan Mean Accuracy</h2>

        <div class="narasi">
            <p>Tabel berikut menyajikan peringkat seluruh skenario model berdasarkan nilai
            <strong>Mean Validation Accuracy</strong> pada lima fold, diurutkan dari yang tertinggi.</p>
        </div>

        {ranking_html_100}

        <h2>📋 Detail Hasil Per Fold</h2>

        <div class="narasi">
            <p>Tabel ini memperlihatkan <strong>Best Validation Accuracy</strong> masing-masing skenario pada
            tiap fold. Perhatikan variasi nilai antar fold — nilai <strong>Standard Deviation</strong> yang
            besar mengindikasikan performa yang tidak stabil, yang semakin diperkuat oleh indikasi overfitting
            pada 100 epoch.</p>
        </div>

        {detail_html_100}

        <footer>
            <p>📅 Eksperimen: <strong>100 Epochs</strong> | Stratified 5-Fold Cross Validation</p>
            <p>🔬 AttentiveSkel-3D Weight Training Classification</p>
            <p>💻 Adam Optimizer (lr=1e-3) | Batch Size=16 | ReduceLROnPlateau</p>
        </footer>

    </div>
</body>
</html>"""

HTML_OUTPUT_100 = ROOT_DIR / "presentasi_kfold_100epochs.html"

with open(HTML_OUTPUT_100, "w", encoding="utf-8") as f:
    f.write(html_100)

print("\n✅ Single-HTML Presentation (100 Epoch) berhasil dibuat!\n")
print(f"🌐 File HTML : {HTML_OUTPUT_100}")
print(f"📦 Ukuran    : {HTML_OUTPUT_100.stat().st_size / 1024:.2f} KB")
print("\n💡 Buka file HTML di browser untuk melihat presentasi visual lengkap.")


✅ Single-HTML Presentation (100 Epoch) berhasil dibuat!

🌐 File HTML : D:\Data-Aji\KULIAH\Tugas-Akhir\AttentiveSkel3D-WeightTraining-PoC\presentasi_kfold_100epochs.html
📦 Ukuran    : 8.38 KB

💡 Buka file HTML di browser untuk melihat presentasi visual lengkap.


# 9. Analisis Komparatif: 50 Epoch vs 100 Epoch

Bagian ini membandingkan hasil evaluasi **50 epoch** dan **100 epoch** secara side-by-side untuk mendukung analisis **BAB 4 Skripsi**.

Analisis mencakup:

- **Delta Mean Accuracy** — Apakah akurasi naik atau turun dengan penambahan epoch?
- **Delta Std Deviation** — Apakah stabilitas model membaik atau memburuk (sinyal overfitting)?
- **Ekspor CSV** — Tabel komparasi disimpan untuk dokumentasi penelitian.
- **Presentasi HTML** — File visual standalone siap dibagikan.

In [ ]:
from src.models.compare_epochs import compare_epochs

comparison_df = compare_epochs(
    csv_50  = ROOT_DIR / "data" / "processed" / "kfold_50epochs_ranking.csv",
    csv_100 = ROOT_DIR / "data" / "processed" / "kfold_100epochs_ranking.csv",
    csv_out = ROOT_DIR / "data" / "processed" / "kfold_comparison_50_vs_100.csv",
    html_out= ROOT_DIR / "presentasi_komparasi_epoch.html",
    verbose = True,
)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Konversi ke persen untuk display
disp = comparison_df.copy()
for col in disp.columns:
    if col != "Skenario":
        disp[col] = (disp[col] * 100).round(3)

disp = disp.rename(columns={
    "Mean Acc (50 Ep)":  "Mean Acc\n50 Ep (%)",
    "Mean Acc (100 Ep)": "Mean Acc\n100 Ep (%)",
    "Std Dev (50 Ep)":   "Std Dev\n50 Ep (%)",
    "Std Dev (100 Ep)":  "Std Dev\n100 Ep (%)",
    "Delta Mean Acc":    "Δ Mean Acc (%)",
    "Delta Std Dev":     "Δ Std Dev (%)",
})

def _color_delta_mean(val):
    color = "#00c853" if val >= 0 else "#d32f2f"
    return f"color: {color}; font-weight: bold"

def _color_delta_std(val):
    color = "#d32f2f" if val > 0 else "#00c853"
    return f"color: {color}; font-weight: bold"

styled_comp = (
    disp.style
    .applymap(_color_delta_mean, subset=["Δ Mean Acc (%)"])
    .applymap(_color_delta_std,  subset=["Δ Std Dev (%)"])
    .background_gradient(cmap="Blues",  subset=["Mean Acc\n50 Ep (%)", "Mean Acc\n100 Ep (%)"], vmin=90, vmax=100)
    .background_gradient(cmap="Reds_r", subset=["Std Dev\n50 Ep (%)",  "Std Dev\n100 Ep (%)"],  vmin=0,  vmax=10)
    .format("{:.3f}%", subset=disp.columns[disp.columns != "Skenario"])
    .set_caption("Komparasi Performa 50 Epoch vs 100 Epoch — Best Val Accuracy per Skenario")
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size","14px"),("font-weight","bold"),("text-align","left"),("padding","8px 0")]},
        {"selector": "th",
         "props": [("font-size","11px"),("text-align","center"),("padding","8px 12px"),("white-space","pre")]},
        {"selector": "td",
         "props": [("font-size","12px"),("text-align","center"),("padding","8px 12px")]},
        {"selector": "td:nth-child(2)",
         "props": [("text-align","left"),("font-weight","600"),("padding-left","14px"),("white-space","nowrap")]},
    ])
)

display(styled_comp)